# Explore input data

Quick look at the post-ingest input parquet at `00-data/current/input.parquet`.
Source schema and renaming live in `scripts/ingest/01_prepare_input_data.py`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INPUT = Path('/mnt/team/idd/pub/idd_tc_mortality/00-data/current/input.parquet')
df = pd.read_parquet(INPUT)
print(f'Source: {INPUT.resolve()}')
print(f'rows: {len(df):,}  cols: {len(df.columns)}')

In [ ]:
df.dtypes

In [ ]:
df.head()

## Coverage: years, basins, islands

In [ ]:
df['year'].agg(['min', 'max', 'nunique'])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
df['year'].value_counts().sort_index().plot.bar(ax=ax)
ax.set_title('Rows per year')
ax.set_xlabel('year'); ax.set_ylabel('rows')
plt.tight_layout(); plt.show()

In [ ]:
# basin distribution; verify literal 'NA' is preserved as a string, not lost to NaN
vc = df['basin'].value_counts(dropna=False)
n_na_string = (df['basin'] == 'NA').sum()
n_basin_nan = df['basin'].isna().sum()
print(vc)
print(f"\nbasin == 'NA' (literal string): {n_na_string:,}")
print(f"basin is NaN: {n_basin_nan}")

In [ ]:
df['is_island'].value_counts()

## Deaths

The S1 hurdle is `P(deaths >= 1)`, so the zero-mass and small counts matter.

In [ ]:
n = len(df)
n_zero = (df['deaths'] == 0).sum()
n_pos  = (df['deaths'] >= 1).sum()
print(f'rows: {n:,}')
print(f'deaths == 0: {n_zero:,} ({n_zero / n:.1%})')
print(f'deaths >= 1: {n_pos:,} ({n_pos / n:.1%})')
print()
df['deaths'].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(np.log10(df.loc[df['deaths'] >= 1, 'deaths']), bins=40)
ax.set_xlabel('log10(deaths) | deaths >= 1')
ax.set_ylabel('rows')
ax.set_title(f'Distribution of positive death counts (n = {n_pos:,})')
plt.tight_layout(); plt.show()

## Continuous covariates: exposed, wind_speed, sdi

In [ ]:
df[['exposed', 'wind_speed', 'sdi']].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].hist(np.log10(df['exposed'].clip(lower=1)), bins=40)
axes[0].set_xlabel('log10(exposed)'); axes[0].set_title('exposed (log10)')
axes[1].hist(df['wind_speed'].dropna(), bins=40)
axes[1].set_xlabel('wind_speed'); axes[1].set_title('wind_speed')
axes[2].hist(df['sdi'].dropna(), bins=40)
axes[2].set_xlabel('sdi'); axes[2].set_title('sdi')
plt.tight_layout(); plt.show()

## Per-storm summary

In [ ]:
per_storm = (df.groupby('storm_id')
               .agg(n_locations=('location_id', 'nunique'),
                    total_deaths=('deaths', 'sum'),
                    total_exposed=('exposed', 'sum'),
                    max_wind=('wind_speed', 'max'),
                    basin=('basin', 'first'),
                    year=('year', 'first')))
print(f'distinct storms: {len(per_storm):,}')
per_storm.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
per_storm.groupby('year').size().plot.bar(ax=ax)
ax.set_title('Distinct storms per year'); ax.set_xlabel('year'); ax.set_ylabel('storms')
plt.tight_layout(); plt.show()

## Cross-tabs

In [ ]:
by_basin = df.groupby('basin').agg(
    rows=('deaths', 'size'),
    nonzero_deaths=('deaths', lambda s: int((s >= 1).sum())),
    total_deaths=('deaths', 'sum'),
    median_exposed=('exposed', 'median'),
)
by_basin['p_nonzero'] = by_basin['nonzero_deaths'] / by_basin['rows']
by_basin.sort_values('rows', ascending=False)

In [ ]:
by_island = df.groupby('is_island').agg(
    rows=('deaths', 'size'),
    nonzero_deaths=('deaths', lambda s: int((s >= 1).sum())),
    total_deaths=('deaths', 'sum'),
)
by_island['p_nonzero'] = by_island['nonzero_deaths'] / by_island['rows']
by_island